New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import os, sys, nbformat
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
from ai_data_scientist import project_manager as pm
handle = pm.resolve_project('kaggle-v020-kaggle-exp-05-diamonds')
pm.ensure_notebook(handle)
print('Notebook:', handle.notebook_path)
print('既存セル数:', len(nbformat.read(handle.notebook_path, as_version=4).cells))


Notebook: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-05-diamonds/notebooks/kaggle-v020-kaggle-exp-05-diamonds.ipynb
既存セル数: 0


In [2]:
setup_source = r'''from pathlib import Path
import os, sys, json, hashlib, io, contextlib, warnings, inspect
from datetime import datetime, timezone
from dataclasses import asdict
ROOT = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(ROOT / 'src'))
os.environ['AI_DATA_SCIENTIST_PROJECTS_ROOT'] = str(ROOT / 'benchmark/projects')
import nbformat, numpy as np, pandas as pd
from IPython.display import display
from ai_data_scientist import project_manager as pm, lifecycle
from ai_data_scientist.language_router import detect_language
RUN_ID = 'v020-exp-05'
handle = pm.resolve_project('kaggle-v020-kaggle-exp-05-diamonds')
pm.ensure_notebook(handle)
data_dir = pm.ensure_data_dir(handle)
lifecycle.register_run(RUN_ID, handle.notebook_path)
language = detect_language('Diamondsデータの価格の違いを日本語で分析してください')
phase_log = []
@contextlib.contextmanager
def phase(label, writing=False):
    if lifecycle.is_cancel_requested(RUN_ID):
        raise RuntimeError('実行の取消が要求されています')
    start = lifecycle.mark_write_start if writing else lifecycle.mark_execution_start
    end = lifecycle.mark_write_end if writing else lifecycle.mark_execution_end
    start(RUN_ID)
    phase_log.append({'phase': label, 'event': 'start', 'utc': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except Exception:
        lifecycle.mark_failed(RUN_ID)
        phase_log.append({'phase': label, 'event': 'failed', 'utc': datetime.now(timezone.utc).isoformat()})
        raise
    finally:
        end(RUN_ID)
        phase_log.append({'phase': label, 'event': 'end', 'utc': datetime.now(timezone.utc).isoformat()})
print({'run_id': RUN_ID, 'language': language, 'state': asdict(lifecycle.get_run_status(RUN_ID))})
'''
api_source = r'''from contextlib import redirect_stdout, redirect_stderr
with phase('Kaggle公開データ検索・取得'):
    # SDK出力や例外本文に認証情報が混入しないよう表示を制限する。
    try:
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi()
            api.authenticate()
            found = api.dataset_list(search='diamonds', page=1)
        refs = [str(item.ref) for item in found]
        slug = 'shivam2503/diamonds'
        print('Kaggle検索結果(refのみ):', refs)
        if slug not in refs:
            with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
                exact = api.dataset_list(search='shivam2503 diamonds', page=1)
            refs += [str(item.ref) for item in exact]
        if slug not in refs:
            raise LookupError('指定slugをKaggle API検索で確認できないため外部フォールバックせず停止')
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            listing = api.dataset_list_files(slug)
            metadata = api.dataset_view(slug)
        file_names = [str(f.name) for f in listing.files]
        print('slug:', slug, 'files:', file_names)
        if 'diamonds.csv' not in file_names:
            raise LookupError('diamonds.csvがAPIファイル一覧にないため停止')
        csv_path = data_dir / 'diamonds.csv'
        with redirect_stdout(io.StringIO()), redirect_stderr(io.StringIO()):
            api.dataset_download_file(slug, 'diamonds.csv', path=str(data_dir), force=True, quiet=True)
        if not csv_path.exists():
            import zipfile
            archive = data_dir / 'diamonds.csv.zip'
            if not archive.exists():
                raise FileNotFoundError('取得ファイルが存在しない')
            with zipfile.ZipFile(archive) as zf:
                names = zf.namelist()
                if names != ['diamonds.csv']:
                    raise ValueError('想定外のZIP内容')
                csv_path.write_bytes(zf.read('diamonds.csv'))
        provenance = {
            'owner_dataset_slug': slug, 'filename': 'diamonds.csv',
            'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
            'sha256': hashlib.sha256(csv_path.read_bytes()).hexdigest(),
            'source_url': 'https://www.kaggle.com/datasets/' + slug,
        }
        description = str(getattr(metadata, 'description', '') or '')
        print('PROVENANCE=' + json.dumps(provenance, ensure_ascii=False))
        print('Kaggle APIデータ説明:\n' + description[:16000])
        print('SDK dataset_download_file署名:', inspect.signature(api.dataset_download_file))
    except Exception as exc:
        lifecycle.mark_failed(RUN_ID)
        raise RuntimeError('Kaggle API取得停止: ' + type(exc).__name__ + '。秘密情報保護のため例外本文は非表示。外部CSV未使用。') from None
'''
intro = '''# Diamonds価格分析 — v020-exp-05

## 分析計画
対象は指定のKaggle公開データのみ。API検索・ファイル一覧・原本取得を先に確認し、SHA-256と取得日時を残す。分析対象の単位・定義の確度をmanifestに分離する。

1. 欠測・重複・ゼロ寸法・非現実的寸法・カテゴリ逸脱を検査し、除外基準と行数を記録する。高価格・大重量だけでは誤りと断定しない。
2. 価格分布、重量と価格の図、品質ごとの無調整集計、Pearson/Spearman関連を調べる。
3. 再現可能な分割でlog価格の重量のみ・品質追加・寸法追加モデルを比較する。重量の非線形性を考慮し、条件調整後の関連と無調整の違いを説明する。
4. 重複・外れ値除外・重量範囲・モデル仕様の感度分析を実施する。結果を読み直して必要な追加依頼を最大3回実行する。
5. 全コードを上から再実行し、根拠manifest、visual_audit=True、lifecycle状態を記録する。

## 因果・適用範囲
観察データ内の記述的・関連的分析であり、処置効果ではない。年代・市場・販売条件・選択過程が不明なら現在の市場価格へ外挿しない。外部CSVミラーは使わない。秘密情報は表示しない。

## 実行経路
分析とKaggle SDK呼出しはJupyter MCPのexecute_cellのみ。ターゲットNotebookの構成・追記はproject_manager.enqueue_writeを用いる。MCPのネイティブ実行・出力保存はtransportが担当する。kernel内から同一kernelへ再入してmcp_gateway.run_and_recordを呼ぶ構成は採用しない（対応する同期MCPClientがkernelに供給されていないため）。偽のexecute/evalクライアントは使わない。
'''
with pm._lock_for(handle.notebook_path):
    pass
# Notebook構成の書込みは公開enqueue_writeで直列化する。
def initial_cells(nb):
    if nb.cells:
        raise RuntimeError('既存ノートブックを上書きしません')
    nb.cells.extend([nbformat.v4.new_markdown_cell(intro), nbformat.v4.new_code_cell(setup_source), nbformat.v4.new_code_cell(api_source)])
pm.enqueue_write(handle, initial_cells)
print('初期セル作成:', handle.notebook_path)


初期セル作成: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-05-diamonds/notebooks/kaggle-v020-kaggle-exp-05-diamonds.ipynb


In [3]:
api_source = api_source.replace('metadata = api.dataset_view(slug)', "metadata_path = api.dataset_metadata(slug, path=str(data_dir))\n        metadata = json.loads(Path(metadata_path).read_text(encoding='utf-8'))")
api_source = api_source.replace("description = str(getattr(metadata, 'description', '') or '')", "description = str(metadata.get('description', '') or '')")
def repair_api(nb):
    nb.cells[2].source = api_source
    nb.cells[2].outputs = []
    nb.cells[2].execution_count = None
    nb.cells[0].source += '\n\n## 外部SDK互換性記録\n初回の取得セルはinstalled KaggleApiにdataset_viewが存在しないためAttributeErrorで停止した（認証情報を含む例外本文は非表示）。dataset_listで対象slugは確認できている。SDKの公開dataset_metadata(dataset, path)に置換して再試行する。Jupytermindの不具合ではなくKaggle SDKメソッド差異。外部ソースへの切替はしていない。失敗状態は初期化セルからの再実行で新しい試行としてregister_runし直す。'
pm.enqueue_write(handle, repair_api)
print('Kaggle SDKの公開metadata取得メソッドへ修正')

Kaggle SDKの公開metadata取得メソッドへ修正


In [4]:
api_source = api_source.replace("description = str(metadata.get('description', '') or '')", "description = str(metadata['info']['description'])")
load_source = r'''from ai_data_scientist.ingestion import SourceSpec, ingest
from ai_data_scientist.cleaning import clean_dataset
from ai_data_scientist.eda import explore
from ai_data_scientist.data_definition import FieldValue, build_manifest
from ai_data_scientist.analysis_assumptions import Assumption, AnalysisAssumptionManifest, check_manifest
from ai_data_scientist.data_quality import detect_anomalies
with phase('データ定義・読込・意味的異常検査'):
    loaded = ingest(SourceSpec('csv', str(csv_path)), row_limit=100000)
    assert not loaded.truncated
    raw = loaded.dataframe
    expected = ['carat','cut','color','clarity','depth','table','price','x','y','z']
    assert set(expected).issubset(raw.columns)
    index_columns = [c for c in raw.columns if c.startswith('Unnamed:')]
    for c in index_columns:
        assert raw[c].is_unique and np.array_equal(raw[c].to_numpy(), np.arange(1, len(raw)+1))
    diamonds = raw[expected].copy()
    source = provenance['source_url'] + ' (Kaggle API dataset_metadata.info.description)'
    def fv(value, status='verified'):
        return FieldValue(value, status, source)
    variable_definitions = {
        'price': {'definition': fv('記録されたダイヤモンド価格'), 'unit': fv('USD'), 'valuation_date': fv(None,'unknown')},
        'carat': {'definition': fv('ダイヤモンド重量'), 'unit': fv('carat (ct)','inferred'), 'mass_conversion': fv('1 ct = 0.2 g; このデータでは変換せず','inferred')},
        'cut': {'definition': fv('カット品質'), 'order': fv(['Fair','Good','Very Good','Premium','Ideal'])},
        'color': {'definition': fv('色評価'), 'order': fv(['J','I','H','G','F','E','D'])},
        'clarity': {'definition': fv('透明度評価'), 'order': fv(['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF'])},
        'x': {'definition': fv('長さ'), 'unit': fv('mm')},
        'y': {'definition': fv('幅'), 'unit': fv('mm')},
        'z': {'definition': fv('高さ/深さ'), 'unit': fv('mm')},
        'depth': {'definition': fv('総深さの割合'), 'unit': fv('%'), 'formula': fv('100 * 2*z/(x+y)','inferred')},
        'table': {'definition': fv('上面幅/最大幅'), 'unit': fv('%','inferred')},
    }
    definition_manifest = build_manifest(
        source={'identity': provenance, 'license': fv(metadata['info']['licenses'][0]['name'], 'unknown'), 'dictionary': fv(description)},
        dataset_scope={'population': fv('約54000個のダイヤモンド'), 'sampling_frame': fv(None,'unknown'), 'observation_period': fv(None,'unknown'), 'measurement_process': fv(None,'unknown')},
        variables=variable_definitions,
        transformations=('CSVの連番を特徴から削除','価格はモデルで自然対数','カテゴリは数値の等間隔得点にしない'))
    print('DATA_DEFINITION_MANIFEST=' + json.dumps(asdict(definition_manifest), ensure_ascii=False))
    print('unknownフィールド:', [(p, asdict(v)) for p,v in definition_manifest.unresolved_fields()])
    print('inferredフィールド:', [(c,k,asdict(v)) for c, fields in variable_definitions.items() for k,v in fields.items() if v.status=='inferred'])
    print('定義上の注意: Kaggleのdepth式はpercentageと記す一方100倍が式から省略されている。値と寸法から下で検査する。carat単位とtableの%は推定でありverifiedとしない。')
    eda_report = explore(diamonds)
    print('EDA欠測:', eda_report.missing_summary)
    print('EDAカテゴリ:', json.dumps(eda_report.categorical_summary, ensure_ascii=False))
    display(diamonds.describe().round(4))
    schema = {c:{'not_null':True} for c in expected}
    schema.update({'carat':{'min':0.01,'max':10,'not_null':True},'price':{'min':1,'not_null':True}, 'depth':{'min':1,'max':100,'not_null':True},'table':{'min':1,'max':100,'not_null':True}})
    for c in ['x','y','z']:
        schema[c]={'min':0.01,'max':20,'not_null':True}
    for c in ['cut','color','clarity']:
        schema[c]={'allowed':variable_definitions[c]['order'].value,'not_null':True}
    anomalies = detect_anomalies(diamonds, schema=schema)
    print('SEMANTIC_ANOMALIES=', [asdict(a) for a in anomalies])
    valid_dims = (diamonds[['x','y','z']] > 0).all(axis=1) & (diamonds[['x','y','z']] <= 20).all(axis=1)
    depth_calculated = 200 * diamonds.z / (diamonds.x + diamonds.y)
    depth_error = (diamonds.depth - depth_calculated).abs()
    print('寸法ゼロ行数:', int((diamonds[['x','y','z']]<=0).any(axis=1).sum()))
    print('寸法>20mm行数:', int((diamonds[['x','y','z']]>20).any(axis=1).sum()))
    print('深さ定義チェック: 有効寸法の絶対差中央値(percentage points)=', float(depth_error[valid_dims].median()), '1pp超=', int((depth_error[valid_dims]>1).sum()))
    display(diamonds.loc[~valid_dims].assign(depth_recomputed=depth_calculated).head(30))
    dedup_report = clean_dataset(diamonds, operation='drop_duplicates')
    complete_report = clean_dataset(dedup_report.dataframe, operation='drop_na')
    base = complete_report.dataframe.copy()
    valid_base = (base[['x','y','z']]>0).all(axis=1) & (base[['x','y','z']]<=20).all(axis=1)
    main = base.loc[valid_base].copy()
    cleaning_summary = {'raw_rows':len(raw),'duplicate_rows_removed':dedup_report.rows_removed,'missing_rows_removed':complete_report.rows_removed,'dimension_flagged_after_dedup':int((~valid_base).sum()),'main_rows':len(main),'dropped_columns':index_columns}
    print('CLEANING=' + json.dumps(cleaning_summary, ensure_ascii=False))
    print('除外理由: 重複レコードは同一観測として扱う仮定。主解析はゼロ/20mm超寸法を除外。価格・重量が大きいだけでは除外せず感度分析する。20mm閾値は物理的確定値ではなく保守的な検査基準。')
    assumptions = AnalysisAssumptionManifest(
        analysis_scope={'target':'当該ファイル内の価格差','outcome':'log(price)','generalization':'現市場への外挿なし'},
        causal_scope='associational', sampling={'n':len(main),'seed':2026,'method':'学習80%/検証20%; 重複除去後'},
        assumptions=(
            Assumption('observational','カテゴリ・重量と価格の関連であり因果効果を主張しない','verified',conclusion_critical=True),
            Assumption('duplicates','同一特徴・価格の重複は同一観測である','assumed',impact_if_false='別商品なら頻度分布が変わる。重複保持の感度分析で確認する',conclusion_critical=True),
            Assumption('dimensions','ゼロ/20mm超の寸法値は主解析から外す','assumed',impact_if_false='正当な極端商品を除外しうる。寸法を使わない全行モデルと比較する',conclusion_critical=True),
            Assumption('representativeness','現在の全ダイヤ市場を代表する','rejected',impact_if_false='現在価格予測への外挿不可',conclusion_critical=False),
        ))
    print('ANALYSIS_ASSUMPTION_MANIFEST=' + json.dumps(asdict(assumptions), ensure_ascii=False))
    print('ASSUMPTION_FINDINGS=', [asdict(f) for f in check_manifest(assumptions)])
    print('非適用: 独立した第二データを指定・取得していないためvalidate_anomalies/compare_datasetsは非実施。同じCSVの複製を独立検証とみなさない。因果推定・時系列分析は識別条件/時点がないため非適用。')
'''
eda_source = r'''from ai_data_scientist.stats_analysis import correlation
from scipy.stats import spearmanr
with phase('初回探索・相関'):
    pearson = correlation(main, 'carat', 'price', language='ja')
    spearman = spearmanr(main.carat, main.price)
    print('PEARSON=' + repr(pearson.statistic))
    print(pearson.interpretation)
    print('SPEARMAN=' + repr(float(spearman.statistic)))
    print('参考: p値は大標本で小さくなる。効果量・分布・検証誤差を重視。相関は因果の証拠ではない。')
    display(main[['carat','price','x','y','z','depth','table']].corr().round(4))
    raw_quality = {}
    for c in ['cut','color','clarity']:
        t = main.groupby(c).agg(n=('price','size'),median_price_USD=('price','median'),mean_carat=('carat','mean'),median_carat=('carat','median'))
        t = t.reindex(variable_definitions[c]['order'].value)
        raw_quality[c] = t
        print('無調整集計:', c)
        display(t.round(4))
    print('PRICE_MEDIAN_USD=' + repr(float(main.price.median())))
'''
chart_setup = r'''from ai_data_scientist.visualization import render_chart, build_image_output
with phase('図表描画準備'):
    def show_chart(frame, kind, x, y, title, xlabel, ylabel):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            png = render_chart(frame, kind=kind, x=x, y=y, title=title, xlabel=xlabel, ylabel=ylabel)
        glyph_warnings = [str(w.message) for w in caught if 'Glyph' in str(w.message) and 'missing' in str(w.message)]
        if glyph_warnings:
            raise RuntimeError('図表の欠落グリフ: ' + '; '.join(glyph_warnings))
        output = build_image_output(png)
        display(output['data'], raw=True)
        print('図表検査: PNG bytes=', len(png), '; missing glyph warnings=', len(glyph_warnings), '; title/xlabel/ylabel明記')
'''
chart_specs = [
    (r'''with phase('価格分布図'):
    show_chart(main, 'hist', 'price', None, '価格分布（高価格だけでは異常としない）', '価格（USD）', 'ダイヤモンド数')
''', {'title':'価格分布（高価格だけでは異常としない）','xlabel':'価格（USD）','ylabel':'ダイヤモンド数','legend':'単一系列: 主解析対象','missing_glyphs':[]}),
    (r'''with phase('重量と価格図'):
    plot_sample = main.sample(min(6000,len(main)), random_state=2026)
    show_chart(plot_sample, 'scatter', 'carat', 'price', '重量と価格（固定seedの6000件標本）', '重量（ct; 単位は推定）', '価格（USD）')
''', {'title':'重量と価格（固定seedの6000件標本）','xlabel':'重量（ct; 単位は推定）','ylabel':'価格（USD）','legend':'各点: ダイヤモンド; seed=2026','missing_glyphs':[]}),
    (r'''with phase('無調整カット図'):
    cut_plot = raw_quality['cut'].reset_index()
    show_chart(cut_plot, 'bar', 'cut', 'median_price_USD', 'カット別価格中央値（重量は未調整）', 'カット品質', '価格中央値（USD）')
''', {'title':'カット別価格中央値（重量は未調整）','xlabel':'カット品質','ylabel':'価格中央値（USD）','legend':'median_price_USD','missing_glyphs':[]}),
]
def add_first_analysis(nb):
    nb.cells[2].source = api_source
    nb.cells.append(nbformat.v4.new_markdown_cell('## データ定義・前提と品質検査\nAPIの原本説明を優先する。未確認の単位・母集団・時点はunknown/inferredのまま記録する。価格範囲の端を機械的に除外せず、明らかな寸法不整合と統計的な裾を分離する。'))
    for src in [load_source, eda_source, chart_setup]:
        nb.cells.append(nbformat.v4.new_code_cell(src))
    for src, meta in chart_specs:
        nb.cells.append(nbformat.v4.new_code_cell(src, metadata={'chart':meta}))
pm.enqueue_write(handle, add_first_analysis)
print('定義・EDA・3図セルを追記')

定義・EDA・3図セルを追記


In [5]:
model_source = r'''import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from threadpoolctl import threadpool_limits
with phase('初回モデル比較・調整後関連'):
    model_frame = main.assign(logprice=np.log(main.price))
    train, test = train_test_split(model_frame, test_size=0.2, random_state=2026)
    quality_terms = {
        'cut': "C(cut, Treatment(reference='Fair'))",
        'color': "C(color, Treatment(reference='J'))",
        'clarity': "C(clarity, Treatment(reference='I1'))",
    }
    def weight_term(df_spline=6):
        return f'bs(np.log(carat), df={df_spline}, degree=3, lower_bound=np.log(0.2), upper_bound=np.log(5.01))'
    def quality_formula(df_spline=6, omit=None):
        return 'logprice ~ ' + weight_term(df_spline) + ' + ' + ' + '.join(v for k,v in quality_terms.items() if k!=omit)
    formulas = {
        '重量log直線': 'logprice ~ np.log(carat)',
        '重量スプライン': 'logprice ~ ' + weight_term(),
        '重量+品質': quality_formula(),
        '重量+品質+寸法': quality_formula() + ' + depth + table + np.log(x) + np.log(y) + np.log(z)',
        '品質のみ': 'logprice ~ ' + ' + '.join(quality_terms.values()),
    }
    fitted, predictions, score_rows = {}, {}, []
    with threadpool_limits(limits=2):
        for label, formula in formulas.items():
            fit = smf.ols(formula, data=train).fit()
            pred_log = np.asarray(fit.predict(test))
            smear = float(np.exp(fit.resid).mean())
            pred_usd = np.exp(pred_log) * smear
            fitted[label] = fit
            predictions[label] = pred_log
            score_rows.append({'モデル':label,'n_train':len(train),'n_test':len(test),
                'test_R2_logprice':r2_score(test.logprice,pred_log),
                'test_RMSE_logprice':float(np.sqrt(np.mean((test.logprice-pred_log)**2))),
                'test_R2_USD':r2_score(test.price,pred_usd),
                'test_MAE_USD':mean_absolute_error(test.price,pred_usd),
                'smearing_factor':smear})
        quality_fit = smf.ols(quality_formula(), data=model_frame).fit(cov_type='HC3')
        ablation_rows = []
        full_rmse = score_rows[2]['test_RMSE_logprice']
        for omitted in ['cut','color','clarity']:
            f = smf.ols(quality_formula(omit=omitted), data=train).fit()
            p = np.asarray(f.predict(test))
            rmse = float(np.sqrt(np.mean((test.logprice-p)**2)))
            ablation_rows.append({'除いた特徴群':omitted,'test_RMSE_logprice':rmse,'増加_logRMSE':rmse-full_rmse})
    model_scores = pd.DataFrame(score_rows)
    ablation = pd.DataFrame(ablation_rows)
    print('MODEL_SCORES=' + model_scores.to_json(orient='records',force_ascii=False))
    display(model_scores.round(5))
    print('BLOCK_ABLATION=' + ablation.to_json(orient='records',force_ascii=False))
    display(ablation.round(5))
    effect_rows = []
    cis = quality_fit.conf_int()
    for col, term in quality_terms.items():
        for level in variable_definitions[col]['order'].value:
            key = term + '[T.' + level + ']'
            coef = float(quality_fit.params.get(key, 0.0))
            lo, hi = (cis.loc[key].to_numpy() if key in cis.index else [0.,0.])
            effect_rows.append({'feature':col,'level':level,'vs_reference':variable_definitions[col]['order'].value[0],
                'adjusted_pct':100*np.expm1(coef),'HC3_lo_pct':100*np.expm1(lo),'HC3_hi_pct':100*np.expm1(hi)})
    effects = pd.DataFrame(effect_rows)
    print('ADJUSTED_EFFECTS=' + effects.to_json(orient='records',force_ascii=False))
    display(effects.round(3))
    ideal_pct = float(effects.query("feature=='cut' and level=='Ideal'").adjusted_pct.iloc[0])
    print('IDEAL_FAIR_ADJUSTED_PCT=' + repr(ideal_pct))
    print('QUALITY_LOG_R2=' + repr(float(model_scores.loc[model_scores['モデル']=='重量+品質','test_R2_logprice'].iloc[0])))
    print('解釈: 係数の%は同じ重量・他品質のモデル上の幾何平均価格比。算術平均価格差や因果効果ではない。HC3区間はモデル内不確実性のみ。対数R2とUSD R2を区別。smearingは訓練残差だけで算出しUSDへの逆変換バイアスを部分補正。')
'''
sens_source = r'''from ai_data_scientist.sensitivity import SensitivityPlan, run_sensitivity
with phase('仕様感度分析'):
    sensitivity_details = []
    def sensitivity_analysis(subset, df_spline):
        pool = {'主解析':main, '重複保持':diamonds.loc[valid_dims], '寸法異常保持':base,
                '価格両端0.5%除外':main.loc[main.price.between(main.price.quantile(.005),main.price.quantile(.995))]}[subset]
        frame = pool.assign(logprice=np.log(pool.price))
        with threadpool_limits(limits=2):
            fit = smf.ols(quality_formula(df_spline=df_spline), data=frame).fit()
        key = quality_terms['cut'] + '[T.Ideal]'
        premium_pct = float(100*np.expm1(fit.params[key]))
        sensitivity_details.append({'subset':subset,'df_spline':df_spline,'n':len(frame),'ideal_fair_pct':premium_pct})
        return premium_pct
    sensitivity_plan = SensitivityPlan(parameter_grid={'subset':['主解析','重複保持','寸法異常保持','価格両端0.5%除外'], 'df_spline':[6,4,8]},max_runs=12)
    sensitivity_report = run_sensitivity(sensitivity_plan, sensitivity_analysis, stability_tolerance=.2)
    print('SENSITIVITY_REPORT=' + json.dumps(asdict(sensitivity_report),ensure_ascii=False))
    display(pd.DataFrame(sensitivity_details).round(4))
    print('stable=', sensitivity_report.stable, '; max_relative_deviation=', sensitivity_report.max_relative_deviation)
    print('判定対象はIdeal/Fairの調整後関連の大きさ。符号一致だけで全結果が安定とは主張しない。許容差20%は価格比の%差に対する相対偏差で事前指定。')
    assumption_status = {'duplicates':'tested','dimensions':'tested'}
    from dataclasses import replace
    assumptions_final = replace(assumptions, assumptions=tuple(replace(a,status=assumption_status.get(a.id,a.status)) for a in assumptions.assumptions))
    print('ANALYSIS_ASSUMPTION_MANIFEST_UPDATED=' + json.dumps(asdict(assumptions_final),ensure_ascii=False))
    print('ASSUMPTION_FINDINGS_UPDATED=', [asdict(f) for f in check_manifest(assumptions_final)])
'''
adjusted_chart_source = r'''with phase('調整後カット価格比図'):
    cut_effects = effects.query("feature=='cut'").copy()
    show_chart(cut_effects, 'bar', 'level', 'adjusted_pct', '重量・色・透明度調整後のカット価格比', 'カット品質（基準Fair）', '幾何平均価格の差（%）')
'''
model_intro = '''## 初回モデルと感度分析
80/20固定分割（seed=2026）、価格は自然対数。重量の非線形性を3次Bスプラインで調整し、品質はダミー変数で扱う。対数R2は対数価格の分散説明率であってUSD価格の分散説明率ではない。寸法群は重量と強く共線的なので「独立した重要特徴」と断定せず、追加したときの検証誤差を比較する。

主解析の全件モデルで条件調整後の品質関連とHC3区間を算出する。感度分析は12仕様（スプライン自由度4/6/8 × 主解析/重複保持/寸法異常保持/価格0.5%裾除外）。主結果と同じ検証セットを用いた特徴群除去は説明的な追加寄与であり、特徴の因果的重要度ではない。
'''
def add_model_cells(nb):
    nb.cells.extend([nbformat.v4.new_markdown_cell(model_intro),nbformat.v4.new_code_cell(model_source),nbformat.v4.new_code_cell(sens_source),nbformat.v4.new_code_cell(adjusted_chart_source,metadata={'chart':{'title':'重量・色・透明度調整後のカット価格比','xlabel':'カット品質（基準Fair）','ylabel':'幾何平均価格の差（%）','legend':'adjusted_pct','missing_glyphs':[]}})])
pm.enqueue_write(handle, add_model_cells)
print('モデル・感度・調整後図を追記')

モデル・感度・調整後図を追記


In [6]:
iteration1_request = '全体モデルのIdeal対Fair約18%という差を、すべての重量に共通する値と解釈してよいか確認してください。重量帯ごとに色・透明度と帯内重量を調整した差、比較群の件数、信頼区間を示し、重量と品質の交互作用を許したモデルの検証誤差も比較してください。疎な組合せや外挿は明示してください。'
iteration1_source = r'''with phase('反復1: 重量帯別の品質差・交互作用'):
    iteration1_request = '全体モデルのIdeal対Fair約18%という差を、すべての重量に共通する値と解釈してよいか確認してください。重量帯ごとに色・透明度と帯内重量を調整した差、比較群の件数、信頼区間を示し、重量と品質の交互作用を許したモデルの検証誤差も比較してください。疎な組合せや外挿は明示してください。'
    bands = [(0.2,.5,'0.2–<0.5'),(.5,1,'0.5–<1'),(1,1.5,'1–<1.5'),(1.5,2,'1.5–<2'),(2,5.02,'2–5.01')]
    strata_rows = []
    with threadpool_limits(limits=2):
        for low,high,label in bands:
            g = model_frame.loc[model_frame.carat.ge(low)&model_frame.carat.lt(high)]
            counts = g.cut.value_counts()
            if min(int(counts.get('Fair',0)),int(counts.get('Ideal',0))) < 50:
                print('帯',label,'はFair/Idealが50件未満で推定を保留:',counts.to_dict())
                continue
            f = smf.ols(quality_formula(df_spline=4),data=g).fit(cov_type='HC3')
            key = quality_terms['cut']+'[T.Ideal]'
            lo,hi=f.conf_int().loc[key].to_numpy()
            strata_rows.append({'重量帯_ct':label,'n':len(g),'n_Fair':int(counts['Fair']),'n_Ideal':int(counts['Ideal']),
                'ideal_fair_pct':100*np.expm1(f.params[key]),'HC3_lo_pct':100*np.expm1(lo),'HC3_hi_pct':100*np.expm1(hi)})
        interaction_formula = quality_formula() + ' + ' + ' + '.join('np.log(carat):'+v for v in quality_terms.values())
        interaction_fit = smf.ols(interaction_formula,data=train).fit()
        interaction_prediction = np.asarray(interaction_fit.predict(test))
        interaction_log_rmse = float(np.sqrt(np.mean((test.logprice-interaction_prediction)**2)))
    strata = pd.DataFrame(strata_rows)
    print('ITERATION1_STRATA=' + strata.to_json(orient='records',force_ascii=False))
    display(strata.round(4))
    print('INTERACTION_LOG_RMSE=' + repr(interaction_log_rmse))
    print('ADDITIVE_LOG_RMSE=' + repr(float(model_scores.loc[2,'test_RMSE_logprice'])))
    print('比較カテゴリの同時組合せの共通支持は全帯で保証されない。帯内OLSでも未観測品質の組合せへ外挿しうる。')
    support = pd.crosstab(pd.cut(model_frame.carat,[.199,.5,1,1.5,2,5.02],right=False),model_frame.cut)
    display(support)
    print('残る限界: 帯内で色・透明度を同時調整しても販売条件等の未観測差は残る。交互作用は追加検証セットで新しく確認した仮説で、事前仮説ではない。')
'''
iteration1_md = '## 反復依頼履歴\n\n### 反復1 — 依頼原文\n'+iteration1_request+'\n\n### 選定理由\n初回の条件調整モデルは品質差を重量にかかわらず一定と仮定している。全体平均の約18%が重量帯で変わるなら結論の適用範囲が変わるため、追加価値がある。初回感度分析はスプライン・除外基準だけで、この仮定をまだ検証していない。\n\n### 結果・証拠・残る限界の記録方法\n直後のITERATION1_STRATA、交互作用RMSE、比較群件数を実行結果として残し、統合Insightから実行済み根拠を引用する。'
def add_iter1(nb):
    nb.cells.extend([nbformat.v4.new_markdown_cell(iteration1_md),nbformat.v4.new_code_cell(iteration1_source)])
pm.enqueue_write(handle,add_iter1)
print('反復1依頼原文と実行コードを記録')

反復1依頼原文と実行コードを記録


In [7]:
old_line = "interaction_formula = quality_formula() + ' + ' + ' + '.join('np.log(carat):'+v for v in quality_terms.values())"
new_line = "interaction_terms = [f\"I(np.log(carat) * ({c} == {level!r}))\" for c in quality_terms for level in variable_definitions[c]['order'].value[1:]]\n        interaction_formula = quality_formula() + ' + ' + ' + '.join(interaction_terms)"
iteration1_source = iteration1_source.replace(old_line,new_line).replace("interaction_prediction = np.asarray(interaction_fit.predict(test))", "assert np.linalg.matrix_rank(interaction_fit.model.exog) == interaction_fit.model.exog.shape[1]\n        interaction_prediction = np.asarray(interaction_fit.predict(test))")
iteration2_request = '0.5ct未満ではFairが112件しかなく、調整後差の区間に0が含まれました。全体のカット差と重量帯別の結論が疎な組合せへの外挿で生じていないか、色・透明度が同じで重量が近いFairとIdealの共通支持層だけで価格比を比較してください。また、深さの再計算値が1パーセントポイント超ずれる行を除いて、寸法追加の結論とカット価格比への影響を確認してください。'
iteration2_source = r'''with phase('反復2: 共通支持・寸法整合性'):
    iteration2_request = '0.5ct未満ではFairが112件しかなく、調整後差の区間に0が含まれました。全体のカット差と重量帯別の結論が疎な組合せへの外挿で生じていないか、色・透明度が同じで重量が近いFairとIdealの共通支持層だけで価格比を比較してください。また、深さの再計算値が1パーセントポイント超ずれる行を除いて、寸法追加の結論とカット価格比への影響を確認してください。'
    matched_rows = []
    rng_match = np.random.default_rng(2026)
    for width in [.02,.05]:
        match_frame = model_frame.loc[model_frame.cut.isin(['Fair','Ideal'])].copy()
        match_frame['bin'] = np.floor((match_frame.carat+.000001)/width).astype(int)
        matched_table = match_frame.groupby(['bin','color','clarity','cut']).logprice.agg(['mean','size']).unstack('cut')
        matched_table = matched_table.dropna()
        matched_table = matched_table.loc[(matched_table['size']['Fair']>=2)&(matched_table['size']['Ideal']>=2)]
        def matched_summary(table,label):
            if len(table)<2:
                print('共通支持不足:',width,label,'層数=',len(table))
                return
            difference = (table['mean']['Ideal']-table['mean']['Fair']).to_numpy()
            nf=table['size']['Fair'].to_numpy(); ni=table['size']['Ideal'].to_numpy()
            weight=nf*ni/(nf+ni)
            estimate=float(100*np.expm1(np.average(difference,weights=weight)))
            boot=[]
            for _ in range(1000):
                ids=rng_match.integers(0,len(table),len(table))
                boot.append(100*np.expm1(np.average(difference[ids],weights=weight[ids])))
            lo,hi=np.quantile(boot,[.025,.975])
            matched_rows.append({'幅_ct':width,'範囲':label,'共通層数':len(table),'Fair件数':int(nf.sum()),'Ideal件数':int(ni.sum()),'ideal_fair_pct':estimate,'層bootstrap_lo_pct':lo,'層bootstrap_hi_pct':hi})
        matched_summary(matched_table,'全重量')
        small=matched_table.loc[matched_table.index.get_level_values('bin')*width < .5]
        matched_summary(small,'0.5ct未満')
    matched_results=pd.DataFrame(matched_rows)
    print('ITERATION2_MATCHED=' + matched_results.to_json(orient='records',force_ascii=False))
    display(matched_results.round(4))
    consistent = model_frame.loc[depth_error.reindex(model_frame.index).le(1)].copy()
    consistent_train=train.loc[train.index.intersection(consistent.index)]
    consistent_test=test.loc[test.index.intersection(consistent.index)]
    geometry_check=[]
    with threadpool_limits(limits=2):
        consistent_full = smf.ols(quality_formula(),data=consistent).fit()
        consistent_ideal_pct=float(100*np.expm1(consistent_full.params[quality_terms['cut']+'[T.Ideal]']))
        for label in ['重量+品質','重量+品質+寸法']:
            fit=smf.ols(formulas[label],data=consistent_train).fit()
            pred=np.asarray(fit.predict(consistent_test))
            usd=np.exp(pred)*float(np.exp(fit.resid).mean())
            geometry_check.append({'モデル':label,'n_train':len(consistent_train),'n_test':len(consistent_test),'log_RMSE':float(np.sqrt(np.mean((consistent_test.logprice-pred)**2))),'USD_MAE':mean_absolute_error(consistent_test.price,usd)})
    print('ITERATION2_DEPTH_CHECK=' + json.dumps({'n_consistent':len(consistent),'excluded':len(model_frame)-len(consistent),'ideal_fair_pct':consistent_ideal_pct,'models':geometry_check},ensure_ascii=False))
    print('残る限界: 共通支持推定は比較可能な層だけを対象とし全データとは母集団が違う。重量はビン幅以内で異なり、未観測販売条件も一致させられない。bootstrapは層の再標本化でモデル選択不確実性や市場外挿は含まない。深さ1pp基準は丸め誤差より大きい検査閾値で測定誤りの確定ではない。')
'''
iteration2_md='### 反復2 — 依頼原文\n'+iteration2_request+'\n\n### 選定理由\n反復1で0.5ct未満の差が不確かであり、全体の約18%を全重量に共通する差とする解釈は撤回する。Fairの少ない帯の組合せ外挿を検査する。また意味的検査でdepth再計算差>1ppが77件あったため、寸法群の追加寄与という結論に影響するか検査する。\n\n### 実行上の修正\n交互作用をカテゴリ全水準のlog重量傾きで表現した初回は、共通log重量がスプライン基底と重複してstatsmodelsのrank-deficient警告が出た。基準水準を除いた差分傾きへ置換し、行列フルランクをassertして再実行した。分析式の設計問題でありJupytermindの不具合ではない。\n\n### 証拠\n直前のITERATION1_STRATAと、直後のITERATION2_MATCHED/ITERATION2_DEPTH_CHECK。結果と限界は根拠付き統合Insightにも記録する。'
def add_iter2(nb):
    nb.cells[15].source=iteration1_source
    nb.cells.extend([nbformat.v4.new_markdown_cell(iteration2_md),nbformat.v4.new_code_cell(iteration2_source)])
pm.enqueue_write(handle,add_iter2)
print('交互作用仕様修正・反復2を記録')

交互作用仕様修正・反復2を記録


In [8]:
iteration3_request='初回の特徴群除去では透明度、色、カットの順に追加説明力が大きくなりました。この順位と品質追加による誤差改善が分割seedを変えても維持されるか確認してください。重量の丸い境界0.5、1.0、1.5、2.0ctで価格の段差を許す仕様も比較し、全体カット価格比の仕様依存性、検証重量帯ごとの誤差、寸法追加の小さな改善の不確実性を示してください。'
iteration3_source=r'''with phase('反復3: 分割・丸い重量境界・誤差の不確実性'):
    iteration3_request='初回の特徴群除去では透明度、色、カットの順に追加説明力が大きくなりました。この順位と品質追加による誤差改善が分割seedを変えても維持されるか確認してください。重量の丸い境界0.5、1.0、1.5、2.0ctで価格の段差を許す仕様も比較し、全体カット価格比の仕様依存性、検証重量帯ごとの誤差、寸法追加の小さな改善の不確実性を示してください。'
    threshold_formula=quality_formula()+' + '+' + '.join(f'I(carat >= {v})' for v in [.5,1.,1.5,2.])
    seed_rows=[]; rank_rows=[]
    with threadpool_limits(limits=2):
        for seed in [2026,17,42]:
            tr,te=train_test_split(model_frame,test_size=.2,random_state=seed)
            score={}
            for label,formula in [('重量のみ',formulas['重量スプライン']),('重量+品質',quality_formula()),('重量+品質+境界',threshold_formula)]:
                f=smf.ols(formula,data=tr).fit()
                p=np.asarray(f.predict(te))
                score[label]=float(np.sqrt(np.mean((te.logprice-p)**2)))
                seed_rows.append({'seed':seed,'モデル':label,'log_RMSE':score[label]})
                if seed==2026 and label=='重量+品質+境界':
                    threshold_prediction=p
            for omit in ['cut','color','clarity']:
                f=smf.ols(quality_formula(omit=omit),data=tr).fit()
                p=np.asarray(f.predict(te))
                rank_rows.append({'seed':seed,'除去':omit,'増加_logRMSE':float(np.sqrt(np.mean((te.logprice-p)**2)))-score['重量+品質']})
        threshold_full=smf.ols(threshold_formula,data=model_frame).fit(cov_type='HC3')
        assert np.linalg.matrix_rank(threshold_full.model.exog)==threshold_full.model.exog.shape[1]
    threshold_ideal_pct=float(100*np.expm1(threshold_full.params[quality_terms['cut']+'[T.Ideal]']))
    print('ITERATION3_SEEDS=' + pd.DataFrame(seed_rows).to_json(orient='records',force_ascii=False))
    print('ITERATION3_RANK=' + pd.DataFrame(rank_rows).to_json(orient='records',force_ascii=False))
    print('THRESHOLD_IDEAL_PCT=' + repr(threshold_ideal_pct))
    display(pd.DataFrame(seed_rows).round(5)); display(pd.DataFrame(rank_rows).round(5))
    band_errors=[]
    test_usd=np.exp(predictions['重量+品質'])*float(np.exp(fitted['重量+品質'].resid).mean())
    for low,high,label in bands:
        mask=test.carat.ge(low)&test.carat.lt(high)
        band_errors.append({'重量帯_ct':label,'n_test':int(mask.sum()),'quality_log_RMSE':float(np.sqrt(np.mean((test.logprice.to_numpy()[mask]-predictions['重量+品質'][mask])**2))),'threshold_log_RMSE':float(np.sqrt(np.mean((test.logprice.to_numpy()[mask]-threshold_prediction[mask])**2))),'quality_USD_MAE':mean_absolute_error(test.price.to_numpy()[mask],test_usd[mask])})
    print('ITERATION3_BAND_ERRORS=' + json.dumps(band_errors,ensure_ascii=False))
    rng_boot=np.random.default_rng(2026)
    log_y=test.logprice.to_numpy()
    bootstrap_rows=[]
    for newer,older in [('重量+品質','重量スプライン'),('重量+品質+寸法','重量+品質')]:
        delta=(log_y-predictions[older])**2-(log_y-predictions[newer])**2
        boot=np.empty(1000)
        for i in range(1000):
            boot[i]=delta[rng_boot.integers(0,len(delta),len(delta))].mean()
        lo,hi=np.quantile(boot,[.025,.975])
        bootstrap_rows.append({'比較':newer+' vs '+older,'MSE改善':float(delta.mean()),'paired_bootstrap_lo':lo,'paired_bootstrap_hi':hi})
    print('ITERATION3_PAIRED_BOOTSTRAP=' + json.dumps(bootstrap_rows,ensure_ascii=False))
    print('残る限界: seed変更は同じデータ内の分割感度で外部検証ではない。反復で同じ検証データを参照しており最終的な未使用評価ではない。段差項は説明的仕様比較で、閾値を越えた因果効果や最適モデルの確定を意味しない。2ct以上は少なく誤差も大きい。bootstrap区間は学習済みモデル固定で、再学習の不確実性は含めない。')
'''
iteration3_md='### 反復3 — 依頼原文\n'+iteration3_request+'\n\n### 選定理由\n反復2では共通支持に限った全体差は約21–22%、小重量帯は2–7%で幅に依存し、強い一律の差は主張できない。一方、主結論の特徴群順位は1分割のみであり、散布図の丸い重量近辺の集積に対して連続スプラインだけでは段差を見落とす可能性がある。寸法追加のlog誤差差が小さいため不確実性も確認する。\n\n### 結果・証拠\n直後のITERATION3_SEEDS/RANK/BAND_ERRORS/PAIRED_BOOTSTRAPとTHRESHOLD_IDEAL_PCTを保存し、統合Insightで根拠を参照する。'
strata_chart=r'''with phase('重量帯別カット差の図'):
    show_chart(strata, 'bar', '重量帯_ct', 'ideal_fair_pct', 'Ideal対Fairの価格比は重量帯で異なる', '重量帯（ct; 区間は隣接表を参照）', '調整後幾何平均価格の差（%）')
'''
ablation_chart=r'''with phase('品質特徴群の追加説明力の図'):
    show_chart(ablation, 'bar', '除いた特徴群', '増加_logRMSE', '特徴群を除いたときの検証誤差増加', '除いた特徴群（因果的重要度ではない）', 'log価格RMSEの増加（無単位）')
'''
def add_iter3(nb):
    nb.cells.extend([nbformat.v4.new_markdown_cell(iteration3_md),nbformat.v4.new_code_cell(iteration3_source)])
    nb.cells.append(nbformat.v4.new_code_cell(strata_chart,metadata={'chart':{'title':'Ideal対Fairの価格比は重量帯で異なる','xlabel':'重量帯（ct; 区間は隣接表を参照）','ylabel':'調整後幾何平均価格の差（%）','legend':'ideal_fair_pct; HC3区間は反復1表参照','missing_glyphs':[]}}))
    nb.cells.append(nbformat.v4.new_code_cell(ablation_chart,metadata={'chart':{'title':'特徴群を除いたときの検証誤差増加','xlabel':'除いた特徴群（因果的重要度ではない）','ylabel':'log価格RMSEの増加（無単位）','legend':'増加_logRMSE','missing_glyphs':[]}}))
pm.enqueue_write(handle,add_iter3)
print('反復3・2図を記録')

反復3・2図を記録


In [9]:
chart_setup = chart_setup.replace('from ai_data_scientist.visualization import render_chart, build_image_output','from ai_data_scientist.visualization import render_chart, build_image_output\nimport matplotlib\nimport matplotlib.pyplot as plt').replace("png = render_chart(frame, kind=kind, x=x, y=y, title=title, xlabel=xlabel, ylabel=ylabel)", "with matplotlib.rc_context({'figure.autolayout': True, 'figure.figsize': (8, 5)}):\n                png = render_chart(frame, kind=kind, x=x, y=y, title=title, xlabel=xlabel, ylabel=ylabel)")
api_source = api_source.replace("description = str(metadata['info']['description'])", "assert provenance['sha256'] == 'fc2f171cc18eae2138d01dcca7179db3bb30ff047dceae4467a056d52133810a', '原本SHA-256が変わったため結果を再評価してください'\n        description = str(metadata['info']['description'])")
# 提出物に残す再現条件とログはターゲットNotebookセルで生成する。
diagnostic_source = r'''from ai_data_scientist.insight_engine import extract_cited_value, record_insight, EvidenceMissingError
with phase('Jupytermind改善候補の再現検査'):
    notebook_snapshot=nbformat.read(handle.notebook_path,as_version=4)
    correlation_cell=next(c for c in notebook_snapshot.cells if c.cell_type=='code' and 'PEARSON=' in c.source)
    stream_text=''.join(o.get('text','') for o in correlation_cell.outputs if o.output_type=='stream')
    stream_value=extract_cited_value({'output':stream_text},r'PEARSON=([0-9.eE+-]+)')
    try:
        record_insight(handle,'stream出力互換性の再現用候補（成功時だけ追記）',correlation_cell.execution_count,stream_value,'associational',language='ja')
    except EvidenceMissingError:
        stream_issue_confirmed=True
        print('JM-02再現: 実行済みstreamにcited_valueが存在するがrecord_insightはEvidenceMissingError。根拠なしのInsightは保存していない。')
    else:
        stream_issue_confirmed=False
        raise AssertionError('想定していたstream非対応が再現しないため改善候補記載を更新してください')
    with matplotlib.rc_context({'figure.autolayout':False,'figure.figsize':(6.4,4.8)}):
        baseline_png=render_chart(strata,kind='bar',x='重量帯_ct',y='ideal_fair_pct',title='Ideal対Fairの価格比は重量帯で異なる',xlabel='重量帯（ct; 区間は隣接表を参照）',ylabel='調整後幾何平均価格の差（%）')
        fig,ax=plt.subplots()
        try:
            strata.plot(kind='bar',x='重量帯_ct',y='ideal_fair_pct',ax=ax)
            ax.set_title('Ideal対Fairの価格比は重量帯で異なる')
            ax.set_xlabel('重量帯（ct; 区間は隣接表を参照）')
            ax.set_ylabel('調整後幾何平均価格の差（%）')
            fig.canvas.draw()
            xlabel_y0=float(ax.xaxis.label.get_window_extent(fig.canvas.get_renderer()).y0)
            print('JM-01再現: デフォルト描画のxlabel下端px=',xlabel_y0,'; canvas範囲はy>=0')
        finally:
            plt.close(fig)
    assert xlabel_y0<0, '既定レイアウトの切れが再現しないため候補を更新してください'
    with phase('改善候補PNG書込み',writing=True):
        baseline_path=data_dir/'jupytermind-layout-reproduction.png'
        baseline_path.write_bytes(baseline_png)
    diagnostic_log={'JM-01':{'xlabel_bottom_pixel':xlabel_y0,'baseline_sha256':hashlib.sha256(baseline_png).hexdigest(),'artifact':str(baseline_path),'workaround':'matplotlib.rc_context figure.autolayout=True, figsize=(8,5)'},'JM-02':{'stream_value':stream_value,'evidence_execution_count':correlation_cell.execution_count,'expected_exception':'EvidenceMissingError','workaround':'IPython display text/plain MIME; extract_cited_valueからrecord_insight'}}
    print('JUPYTERMIND_CANDIDATE_LOG=' + json.dumps(diagnostic_log,ensure_ascii=False))
'''
modules_md='''## 使用したJupytermindモジュール
このNotebookで直接importし、直接呼び出したAPIだけを列挙する。ライブラリ内部の間接利用は使用済みに含めない。

| モジュール | 直接呼出し（クラス構築を含む） | 目的 |
|---|---|---|
| ai_data_scientist.project_manager | resolve_project, ensure_notebook, ensure_data_dir, enqueue_write | slug検証、指定保存先、安定データパス、直列Notebook書込み |
| ai_data_scientist.language_router | detect_language | 日本語での報告 |
| ai_data_scientist.lifecycle | register_run, mark_execution_start/end, mark_write_start/end, is_cancel_requested, mark_completed, mark_failed, get_run_status, wait_for_quiescence | 実行/書込みフェーズ・取消確認・終了状態 |
| ai_data_scientist.ingestion | SourceSpec, ingest | 原本CSVを上限確認つきで読込 |
| ai_data_scientist.cleaning | clean_dataset | 重複・欠測の除去行数記録 |
| ai_data_scientist.eda | explore | 欠測、型、カテゴリ分布、基本統計 |
| ai_data_scientist.stats_analysis | correlation | Pearson係数と日本語解釈 |
| ai_data_scientist.data_definition | FieldValue, build_manifest, DataDefinitionManifest.unresolved_fields | 単位・定義・出典・未解決定義 |
| ai_data_scientist.analysis_assumptions | Assumption, AnalysisAssumptionManifest, check_manifest | 非因果の適用範囲と前提リスク |
| ai_data_scientist.data_quality | detect_anomalies | 意味的制約逸脱 |
| ai_data_scientist.sensitivity | SensitivityPlan, run_sensitivity | 有限12仕様の安定性判定 |
| ai_data_scientist.visualization | render_chart, build_image_output | 日本語図の生成・PNG MIME永続化 |
| ai_data_scientist.insight_engine | extract_cited_value, record_insight（EvidenceMissingErrorを明示処理） | 実行出力の値の抽出・根拠つきInsight |
| ai_data_scientist.notebook_audit | audit_notebook(visual_audit=True) | Notebook実行・根拠・図表の監査 |

KaggleApi、pandas、NumPy、scipy、statsmodels、scikit-learn、matplotlib、nbformat、threadpoolctlは外部ライブラリであり、Jupytermind利用数に含めない。mcp_gateway、dataset_validation、anomaly_detection、ML拡張、独立データvalidate_anomaliesは直接利用していない。ネイティブMCP transportの呼出しはCLIが行い、kernel内に適合MCPClientがないためrun_and_recordは非適用。外部API取込はSDKで取得したローカル原本をingestしており、ingestのAPI fetcher経由ではない。正規分布のz-scoreで高価格を誤りとする方法は価格の歪みのため採用しない。
'''
improvements_md='''## Jupytermind改善候補
候補はコードを修正せず、再現検査と回避策をNotebookに残す。Kaggle SDKのdataset_view非搭載とmetadataのinfoラッパー、statsmodelsの交互作用ランク不足は別の問題であり、この候補には含めない。Copilot CLI・ベンチマークランナーの問題をJupytermindの不具合として扱わない。

| ID・分類 | 再現条件 | 期待する挙動 | 実際の挙動 | 影響 | 回避策 | 関係セル・ログ |
|---|---|---|---|---|---|---|
| JM-01 可視化の可読性/レイアウト不足 | visualization.render_chartの既定6.4×4.8図で、barに長い重量帯ラベルと日本語xlabelを指定 | ラベル全体がPNG内に入り読める | xlabelのbounding box下端が負のpixel座標となり、PNGでxlabelが切れる | 単位・軸の意味が見えない。メタデータとbyte-densityのvisual auditだけでは切れを検出できない | 描画呼出しをrc_context(figure.autolayout=True, figsize=(8,5))で囲む。提出用6図は回避策を適用 | 重量帯別図、描画準備、改善候補再現検査。JUPYTERMIND_CANDIDATE_LOGのxlabel_bottom_pixelとPNG SHA。data/jupytermind-layout-reproduction.png |
| JM-02 evidence出力形式互換性不足 | 実行済み相関セルのprint出力に実際のPEARSON値がある状態で、record_insightへそのexecution_count/valueを渡す | nbformat stream.textも根拠出力として解決する | _find_evidence_cellはdata MIMEだけを検索しEvidenceMissingError | MCPネイティブ実行のprintのみの値ではInsightを記録できない | 計算済み根拠をtext/plain MIMEにdisplayしてからextract_cited_value/record_insightする。根拠内容を捏造しない | 初回探索・相関、改善候補再現検査、統合根拠出力、最終監査。JM-02ログのstream_value、evidence_execution_count、例外クラス |

再現検査は期待されたEvidenceMissingErrorだけを捕捉し、その他のエラーは隠さない。図表監査のbyte-densityは目視やテキスト配置の完全な代替ではないため、実際のMCP表示でも回避後の図を確認する。
'''
stop_md='''## 反復の終了判断と残る限界
3回の追加依頼を実行した。全体の重量関連、品質の追加説明力、透明度>色>カットというモデル内の特徴群順位は確認できた。カット価格比は重量帯・共通支持母集団に依存するため「すべての石で約18%」という表現を採用しない。小重量帯のFairが少ないことは同じデータでモデルを増やしても解消できず、未測定販売条件や市場時点も復元できない。最大3回に到達し、追加モデルを探索して有意な結論を得ようとすることは避ける。外部/未使用データ検証を必要とする限界を残して終了する。
'''
def add_finish_scaffolding(nb):
    nb.cells[2].source=api_source
    nb.cells[6].source=chart_setup
    nb.cells.extend([nbformat.v4.new_markdown_cell(stop_md),nbformat.v4.new_markdown_cell(modules_md),nbformat.v4.new_markdown_cell(improvements_md),nbformat.v4.new_code_cell(diagnostic_source)])
pm.enqueue_write(handle,add_finish_scaffolding)
print('図表レイアウトの回避策・使用モジュール・改善候補を記録')

図表レイアウトの回避策・使用モジュール・改善候補を記録


In [10]:
evidence_source=r'''with phase('統合根拠出力・報告用結果'):
    model_map=model_scores.set_index('モデル')
    evidence_metrics={
        'raw_rows':int(len(raw)), 'main_rows':int(len(main)), 'duplicates_removed':int(dedup_report.rows_removed), 'dimension_excluded':int((~valid_base).sum()),
        'pearson':float(pearson.statistic),'spearman':float(spearman.statistic),
        'weight_log_R2':float(model_map.loc['重量スプライン','test_R2_logprice']),
        'quality_log_R2':float(model_map.loc['重量+品質','test_R2_logprice']),
        'quality_USD_R2':float(model_map.loc['重量+品質','test_R2_USD']),
        'weight_USD_MAE':float(model_map.loc['重量スプライン','test_MAE_USD']),
        'quality_USD_MAE':float(model_map.loc['重量+品質','test_MAE_USD']),
        'fair_median_USD':float(raw_quality['cut'].loc['Fair','median_price_USD']),
        'ideal_median_USD':float(raw_quality['cut'].loc['Ideal','median_price_USD']),
        'ideal_adjusted_pct':ideal_pct,
        'ideal_small_pct':float(strata.iloc[0].ideal_fair_pct),
        'ideal_small_lo_pct':float(strata.iloc[0].HC3_lo_pct),
        'ideal_small_hi_pct':float(strata.iloc[0].HC3_hi_pct),
        'ideal_1to1point5_pct':float(strata.iloc[2].ideal_fair_pct),
        'matched_global_002_pct':float(matched_results.iloc[0].ideal_fair_pct),
        'matched_small_002_pct':float(matched_results.iloc[1].ideal_fair_pct),
        'sensitivity_stable':bool(sensitivity_report.stable),
        'sensitivity_max_relative_deviation':float(sensitivity_report.max_relative_deviation),
        'sensitivity_min_pct':float(min(r.value for r in sensitivity_report.results)),
        'sensitivity_max_pct':float(max(r.value for r in sensitivity_report.results)),
        'threshold_ideal_pct':threshold_ideal_pct,
        'clarity_ablation_logRMSE':float(ablation.set_index('除いた特徴群').loc['clarity','増加_logRMSE']),
        'color_ablation_logRMSE':float(ablation.set_index('除いた特徴群').loc['color','増加_logRMSE']),
        'cut_ablation_logRMSE':float(ablation.set_index('除いた特徴群').loc['cut','増加_logRMSE']),
        'geometry_USD_MAE':float(model_map.loc['重量+品質+寸法','test_MAE_USD']),
        'tail_2ct_USD_MAE':float(band_errors[-1]['quality_USD_MAE']),
        'depth_disagreement_removed':int(len(model_frame)-len(consistent)),
    }
    evidence_text=json.dumps(evidence_metrics,ensure_ascii=False,indent=2)
    display({'text/plain':evidence_text},raw=True)
    print('根拠出力は計算済み変数から生成し、手入力した結果値は含まない。')
    insight_specs=[
        ('main_rows',f'分析対象は原本{len(raw):,}行から重複{dedup_report.rows_removed}行、主解析の寸法フラグ{(~valid_base).sum()}行を除いた{len(main):,}行。欠測除外は{complete_report.rows_removed}行。高価格・大重量だけでは削除していない。価格はKaggle辞書でUSD、寸法はmmと確認済み。重量ctとtableの%は推定、時点・母集団・測定過程・ライセンスは未確認である。','descriptive'),
        ('pearson',f'重量が価格差を最も大きく説明する。重量と価格のPearson相関は{pearson.statistic:.4f}、Spearmanは{spearman.statistic:.4f}。重量スプラインだけで検証log価格R2={evidence_metrics["weight_log_R2"]:.4f}。これは重量を増やす因果効果ではなく、このファイル内の関連である。','associational'),
        ('quality_USD_R2',f'重量だけでは説明しきれない品質差がある。色・透明度・カットを加えると検証log価格R2は{evidence_metrics["weight_log_R2"]:.4f}から{evidence_metrics["quality_log_R2"]:.4f}へ、USD価格R2は{evidence_metrics["quality_USD_R2"]:.4f}となる。USDの平均絶対誤差は{evidence_metrics["weight_USD_MAE"]:.1f}から{evidence_metrics["quality_USD_MAE"]:.1f} USDへ減少。log分散説明率とUSD分散説明率を混同しない。','associational'),
        ('clarity_ablation_logRMSE',f'この仕様の品質群の追加説明力は透明度>色>カット。各群を除くと検証log価格RMSEが順に{evidence_metrics["clarity_ablation_logRMSE"]:.4f}、{evidence_metrics["color_ablation_logRMSE"]:.4f}、{evidence_metrics["cut_ablation_logRMSE"]:.4f}増加し、seed=2026/17/42でも順位を維持した。共線性と分布に依存する特徴群除去評価であり、因果的重要度や品質1段階の共通価格効果ではない。','associational'),
        ('ideal_adjusted_pct',f'無調整のIdeal中央値{evidence_metrics["ideal_median_USD"]:.0f} USDはFairの{evidence_metrics["fair_median_USD"]:.0f} USDより低いが、小重量の構成差がある。同じ重量・色・透明度の加法モデルではIdeal/Fairの幾何平均価格差は約{ideal_pct:.2f}%で符号が逆転する。仕様感度12通りでは{evidence_metrics["sensitivity_min_pct"]:.2f}–{evidence_metrics["sensitivity_max_pct"]:.2f}%。stable={sensitivity_report.stable}、max_relative_deviation={sensitivity_report.max_relative_deviation:.6f}（約{100*sensitivity_report.max_relative_deviation:.2f}%、許容20%）。段差仕様は{threshold_ideal_pct:.2f}%。構成差・条件調整を無視した単純ランキングは避ける。','associational'),
        ('ideal_small_pct',f'反復1で一律約18%という解釈は撤回した。0.2–<0.5ct帯のIdeal/Fair調整差は{evidence_metrics["ideal_small_pct"]:.2f}%、HC3 95%区間は{evidence_metrics["ideal_small_lo_pct"]:.2f}–{evidence_metrics["ideal_small_hi_pct"]:.2f}%で0を含む。Fair={int(strata.iloc[0].n_Fair)}件に対しIdeal={int(strata.iloc[0].n_Ideal)}件で比較が疎。1–<1.5ctは{evidence_metrics["ideal_1to1point5_pct"]:.2f}%。交互作用仕様の検証logRMSEは{interaction_log_rmse:.5f}（加法{float(model_map.loc["重量+品質","test_RMSE_logprice"]):.5f}）。品質差は重量帯依存で、区間はモデル条件下の不確実性である。','associational'),
        ('matched_global_002_pct',f'反復2の同じ色・透明度、重量幅0.02ctの共通支持層では全体Ideal/Fair差は{evidence_metrics["matched_global_002_pct"]:.2f}%（{int(matched_results.iloc[0]["共通層数"])}層）、小重量帯は{evidence_metrics["matched_small_002_pct"]:.2f}%でbootstrap区間に0を含む。幅0.05ctでは小重量差が変わるので小重量の正の差を頑健とはしない。これは共通支持の別母集団で全体OLS18%と同じestimandではない。深さ差>1ppの{len(model_frame)-len(consistent)}行を除いても全体調整差は{consistent_ideal_pct:.2f}%で主要方向は変わらなかった。','associational'),
        ('geometry_USD_MAE',f'寸法は重量と強く相関し、独立寄与を過大解釈しない。寸法群追加のlog誤差改善は小さく、USD MAEは{evidence_metrics["quality_USD_MAE"]:.1f}から{evidence_metrics["geometry_USD_MAE"]:.1f} USDへ悪化した。反復3のpaired bootstrapではlog MSE改善は正だったが、尺度が違うためUSD予測改善とは主張しない。深さ整合性除外後もUSD MAEの改善は確認できない。','associational'),
        ('tail_2ct_USD_MAE',f'反復3では丸い重量境界を許すと3seedで全体log誤差が少し改善したが、1.5–<2ctでは悪化する。品質追加と特徴群順位という主結論は維持。2ct以上は検証{band_errors[-1]["n_test"]}件でUSD MAE={evidence_metrics["tail_2ct_USD_MAE"]:.1f}と大きく、全体MAEだけで大型石の精度を保証できない。同じ検証データを反復参照した探索的評価であり、未使用外部データでの性能とは異なる。','associational'),
    ]
'''
final_source=r'''from ai_data_scientist.notebook_audit import audit_notebook
from importlib.metadata import version
with phase('根拠つきInsightと実行メタデータの書込み',writing=True):
    def remove_generated(nb):
        nb.cells=[c for c in nb.cells if not c.metadata.get('generated_insight',False)]
    pm.enqueue_write(handle,remove_generated)
    snap=nbformat.read(handle.notebook_path,as_version=4)
    evidence_cell=next(c for c in snap.cells if c.metadata.get('role')=='consolidated-evidence')
    actual_text='\n'.join(o.get('data',{}).get('text/plain','') for o in evidence_cell.outputs)
    assert json.loads(actual_text)==evidence_metrics
    for key,text,claim_type in insight_specs:
        cited=extract_cited_value({'output':actual_text},r'"'+key+r'": ([0-9.eE+\-]+)')
        record_insight(handle,text,evidence_cell.execution_count,cited,claim_type,language='ja')
        def tag_latest(nb):
            nb.cells[-1].metadata['generated_insight']=True
            nb.cells[-1].metadata['evidence_cell_id']=evidence_cell.id
        pm.enqueue_write(handle,tag_latest)
    def finalize_metadata(nb):
        final_cell=next(c for c in nb.cells if c.metadata.get('role')=='final-audit')
        nb.cells=[c for c in nb.cells if c.id!=final_cell.id]+[final_cell]
        prior_codes=[c for c in nb.cells[:-1] if c.cell_type=='code']
        counts=[c.execution_count for c in prior_codes]
        assert all(isinstance(n,int) for n in counts)
        assert len(counts)==len(set(counts)) and counts==sorted(counts), '上からの実行順とcountが不整合'
        nb.metadata['diamonds_analysis']={
            'run_id':RUN_ID,'provenance':provenance,'data_definition':asdict(definition_manifest),
            'analysis_assumptions':asdict(assumptions_final),'sensitivity':asdict(sensitivity_report),
            'iterations':[iteration1_request,iteration2_request,iteration3_request],
            'jupytermind_improvement_candidates':diagnostic_log,
            'source_evidence_cells':[{'id':c.id,'execution_count':c.execution_count,'first_line':c.source.splitlines()[0]} for c in prior_codes],
            'versions':{p:version(p) for p in ['kaggle','pandas','numpy','scipy','statsmodels','scikit-learn','matplotlib','nbformat','threadpoolctl']},
            'execution_route':'Jupyter MCP execute_cell; no terminal/subprocess/agent',
            'reexecution_requirement':'kernelをrestartし、全codeセルを上からexecute_cellで実行; SHA一致・実行順を検査',
        }
    pm.enqueue_write(handle,finalize_metadata)
with phase('最終Notebook・図表監査'):
    audit_report=audit_notebook(handle.notebook_path,visual_audit=True)
    print('NOTEBOOK_AUDIT=' + json.dumps({**asdict(audit_report),'ok':audit_report.ok},ensure_ascii=False))
    if not audit_report.ok:
        raise RuntimeError('Notebook監査に未解決エラーがあります')
lifecycle.mark_completed(RUN_ID)
print('LIFECYCLE_COMPLETION=' + json.dumps(asdict(lifecycle.get_run_status(RUN_ID)),ensure_ascii=False))
with phase('監査・ライフサイクルログ永続化',writing=True):
    def persist_status(nb):
        nb.metadata['diamonds_analysis']['notebook_audit']={**asdict(audit_report),'ok':audit_report.ok,'visual_audit':True}
        nb.metadata['diamonds_analysis']['phase_log']=phase_log.copy()
        nb.metadata['diamonds_analysis']['lifecycle']={**asdict(lifecycle.get_run_status(RUN_ID)),'pending_notebook_writes':0,'locks_held':0,'snapshot_timing':'最終書込み完了後に両countが0になることを次行で検査'}
    pm.enqueue_write(handle,persist_status)
quiescent_status=lifecycle.wait_for_quiescence(RUN_ID,timeout_s=5)
assert quiescent_status.state=='completed' and quiescent_status.active_cell_executions==0 and quiescent_status.pending_notebook_writes==0 and quiescent_status.locks_held==0
print('QUIESCENT_STATUS=' + json.dumps(asdict(quiescent_status),ensure_ascii=False))
'''
final_md='''## 統合Insightと再実行・監査
以下の統合根拠セルは計算済み変数をtext/plain MIMEに出力する。根拠値はその保存済み出力からextract_cited_valueで抽出し、record_insightで根拠を検証して記録する。再実行時には古い生成Insightを削除して根拠countを更新し、重複・古いcountを残さない。最終セルはNotebookとPNGをvisual_audit=Trueで監査し、失敗ならfailed、成功ならcompletedを記録する。

再実行手順: 同じ環境とKaggle認証を利用し、kernelを再起動してコードセルを上から実行する。毎回SDKで再検索・再取得し、原本SHA-256が変われば停止する。秘密情報をコードに埋め込まず、環境に設定済みの認証を用いる。データ・分析manifest、全バージョン、実行セルID/count、フェーズログ、監査、状態はNotebook.metadata.diamonds_analysisにも保存する。固定seedの数値が再現し、取得日時のみ更新される設計である。
'''
def add_final(nb):
    nb.cells.extend([nbformat.v4.new_markdown_cell(final_md),nbformat.v4.new_code_cell(evidence_source,metadata={'role':'consolidated-evidence'}),nbformat.v4.new_code_cell(final_source,metadata={'role':'final-audit'})])
pm.enqueue_write(handle,add_final)
print('統合根拠・Insight生成・最終監査セルを追記')

統合根拠・Insight生成・最終監査セルを追記


In [11]:
final_source=final_source.replace("nb.metadata['diamonds_analysis']['lifecycle']={**asdict(lifecycle.get_run_status(RUN_ID)),'pending_notebook_writes':0,'locks_held':0,'snapshot_timing':'最終書込み完了後に両countが0になることを次行で検査'}", "nb.metadata['diamonds_analysis']['status_during_log_write']=asdict(lifecycle.get_run_status(RUN_ID))")
final_source += "\nwith phase('実測完了状態の保存',writing=True):\n    def persist_quiescence(nb):\n        nb.metadata['diamonds_analysis']['lifecycle']=asdict(quiescent_status)\n        nb.metadata['diamonds_analysis']['lifecycle_snapshot_timing']='前のログ書込み完了後の実測値を保存'\n        nb.metadata['diamonds_analysis']['phase_log']=phase_log.copy()\n    pm.enqueue_write(handle,persist_quiescence)\nprint('最終状態:', asdict(lifecycle.get_run_status(RUN_ID)))\n"
def repair_status(nb):
    next(c for c in nb.cells if c.metadata.get('role')=='final-audit').source=final_source
    nb.metadata['kernelspec']={'display_name':'Python 3','language':'python','name':'python3'}
pm.enqueue_write(handle,repair_status)
print('終了状態は実測のquiescent_statusを保存; 現在セル数=',len(nbformat.read(handle.notebook_path,as_version=4).cells))

終了状態は実測のquiescent_statusを保存; 現在セル数= 29


In [12]:
chart_setup=chart_setup.replace('import matplotlib.pyplot as plt', 'import matplotlib.pyplot as plt\nimport japanize_matplotlib\njapanize_matplotlib.japanize()')
font_reproducer=r'''
    with matplotlib.rc_context({'font.family':['DejaVu Sans']}):
        with warnings.catch_warnings(record=True) as font_caught:
            warnings.simplefilter('always')
            render_chart(main.head(100),kind='scatter',x='carat',y='price',title='日本語再現',xlabel='重量',ylabel='価格')
        font_glyph_warnings=[str(w.message) for w in font_caught if 'Glyph' in str(w.message) and 'missing' in str(w.message)]
    assert font_glyph_warnings, '一度適用済みフォントflagとrc_contextの問題が再現しない'
    diagnostic_log['JM-03']={'glyph_warning_count':len(font_glyph_warnings),'sample':font_glyph_warnings[:3],'condition':'日本語を1回render済み; rc_contextでfont.familyをDejaVu Sansに戻して日本語render','workaround':'japanize_matplotlib.japanizeをrc_contextの外側で明示しIPAexGothic設定を保持'}
    print('JM-03再現: font設定をcontextで戻すとrender_chartは再適用せずmissing glyph warnings=',len(font_glyph_warnings))
'''
diagnostic_source=diagnostic_source.replace("    print('JUPYTERMIND_CANDIDATE_LOG=' + json.dumps(diagnostic_log,ensure_ascii=False))",font_reproducer+"    print('JUPYTERMIND_CANDIDATE_LOG=' + json.dumps(diagnostic_log,ensure_ascii=False))")
font_md='''\n\n### JM-03 — 日本語フォント初期化とrc_contextの整合性（可視化不具合候補）
再現条件: 冷スタート後、日本語図をrc_context内で初回render_chartする。最初の図は読めるが、context終了でfont.familyが戻った後、2番目の図をrender_chartするとGlyph missing警告が出る。期待: render_chartが日本語図の度に有効な日本語フォントを保証する。実際: _japanese_font_appliedという一度限りのflagが残り、現在のrcParamsでフォントが戻っていることを再確認しない。影響: 再実行中の散布図が欠落グリフ検査でRuntimeErrorとなりrunはfailedになった。回避策: 外部japanize_matplotlib.japanize()をrc_contextの外で明示適用してから描画し、全図でフォントを維持する。Jupytermind内部のフォント初期化stateとMatplotlib contextの相互作用であり、CLI/Kaggle/runnerとは別問題。
関係セル: 描画準備、重量と価格図、改善候補再現検査。ログ: Notebook.metadata.diamonds_execution_attemptsとJUPYTERMIND_CANDIDATE_LOG.JM-03（warning数と代表3警告）。再現検査はfont.familyを一時的に戻すだけでmodule内部flagを改変しない。最終図はグリフ警告0を要求する。
'''
def repair_fonts(nb):
    errors=[{'cell_id':c.id,'cell_index':i,'execution_count':c.execution_count,'ename':o.ename,'evalue_excerpt':o.evalue[:400]} for i,c in enumerate(nb.cells) if c.cell_type=='code' for o in c.outputs if o.output_type=='error']
    nb.metadata.setdefault('diamonds_execution_attempts',[]).append({'attempt':'cold-replay-1','state':'failed','reason':'rc_context終了で日本語fontが戻る; 描画missing glyph検査で停止','errors':errors})
    nb.cells[6].source=chart_setup
    nb.cells[25].source=diagnostic_source
    nb.cells[24].source+=font_md
pm.enqueue_write(handle,repair_fonts)
print('JM-03を記録し日本語fontをcontext外で明示設定')

JM-03を記録し日本語fontをcontext外で明示設定
